# Partial differential equations: The (parabolic) heat equation
The [heat equation](https://en.wikipedia.org/wiki/Heat_equation) is the typical example of a parabolic partial differential equation. Here we have the time and we are looking for an actual time evolution. It can be written as 

\begin{equation}
\frac{\partial u}{\partial t} = \alpha \nabla^2 u,
\end{equation}
where $u$ represents the temperature, or the concentration (diffusion problem) and so on. The coefficient $\alpha$ takes into account the thermal conductivity, specific heat, and density of the material. 

In the following, we will use as main references the book from Chapra and also 
- https://github.com/numerical-mooc/numerical-mooc
- https://aquaulb.github.io/book_solving_pde_mooc/solving_pde_mooc/notebooks/01_Introduction/01_00_Preface.html
- Wikipedia

In [ ]:
# Import needed utils, like for embeding videos and url
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', 'utils')))
from embed import embed


In [ ]:
embed("https://www.youtube.com/watch?v=ly4S0oi3Yz8")

## The explicit method: Forward time centered space

As is usual in finite differences, we discretize both the spatial (up to second order) and the time (linear order) derivatives to get

\begin{equation}
\frac{u^{n+1}_i - u^n_i}{\Delta t} = \alpha \frac{u_{i+1}^n - 2u_i^n + u_{i-1}^n}{\Delta x^2}, 
\end{equation}
where the superscript corresponds to a step in time, while the subscript corresponds to step in space (we are here solving a one-dimensional equation). This is the so called [FTCS - Forward Time Centered Space](https://en.wikipedia.org/wiki/FTCS_scheme) method, which is an [explicit method](https://en.wikipedia.org/wiki/Explicit_and_implicit_methods) which allows to compute the next (in time) value in terms of the previous one. 

Reorganizing terms  

\begin{equation}
u_{i}^{n+1} = u_{i}^{n}+\frac{\alpha\Delta t}{\Delta x^2}(u_{i+1}^{n}-2u_{i}^{n}+u_{i-1}^{n}) = u_{i}^{n}+\lambda(u_{i+1}^{n}-2u_{i}^{n}+u_{i-1}^{n}),
\end{equation}
which can be visualized as

```{figure} https://upload.wikimedia.org/wikipedia/commons/c/c2/Explicit_method-stencil.svg
:alt: 
:class: bg-primary
:width: 80%
:align: center
```

```{note} Limitations
The explicit ftcs method is $O(N)$ but $\Delta t \propto \Delta x^2$

```


### Numerical stability
By using the [von Neumann stability analysis](https://en.wikipedia.org/wiki/Von_Neumann_stability_analysis) it is possible to show that the solution is stable (errors are not amplified) only if

\begin{equation}
\lambda = \frac{\alpha \Delta t}{\Delta x^2} \le \frac{1}{2}
\end{equation}

or, in 2D with different spacing
\begin{equation}
\lambda = \frac{\alpha \Delta t}{\Delta x^2} \le \frac{1}{2(1 + \frac{\Delta x^2}{\Delta y^2})}.
\end{equation}

This is very important: the discretization cannot be changed arbitrarily, otherwise you will have instabilities. 

### Example: Finding the temperature distribution on a thind rod

Now let's solve an example (Chapra 30.1): A thin rod of 10 cm, with $\alpha = 0.835$ cm$^2$/s, $\Delta t = 0.1 $s, $\Delta x = 10$ cm, and $\lambda = \alpha \Delta t/\Delta x^2 = 0.020875$. The boundary conditions are $T(0) = 100$ C, and $T(10) = 50$ C. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D

# ----------------------------
# 1. Problem Setup
# ----------------------------
L = 10.0
NX = 51                  # Spatial points (fixed ends + interior)
DX = L / (NX - 1)        # dx = L/(N-1) for correct grid spacing
ALPHA = 0.5              # Thermal diffusivity
LAMBDA = 0.45            # Stability parameter (must be <= 0.5 for FTCS)
DT = LAMBDA * DX**2 / ALPHA  # Compute dt from stability limit
N_STEPS = 500

# Grids
x = np.linspace(0, L, NX)
t = np.arange(N_STEPS + 1) * DT

# Initial & Boundary Conditions
T = np.zeros(NX)
T[0] = 100.0             # Left Dirichlet BC
T[-1] = 50.0             # Right Dirichlet BC
T[1:-1] = 20.0           # Uniform interior (try Gaussian for more visual interest)
#T[1:-1] = np.random.uniform(-55.0, 50.0, size = len(T[1:-1]))


In [ ]:
# ----------------------------
# 2. FTCS Solver
# ----------------------------
def ftcs(T, dt, dx, alpha):
    """
    Perform a ftcs step. NOTE: Do not overwrite T, return the new values in Tnew
    Ignore the borders, those are the boundary conditions
    ARGS
    T = Current Temperature array
    deltat = Time step
    deltax = Spatial step
    lambdac = Coefficient parameter
    RETURNS
    Tnew = New temperature
    """
    Tnew = T.copy()
    lambdac = alpha * dt / dx**2
    # YOUR CODE HERE
    pass
    return Tnew

In [ ]:
# Storage for solution history
U = np.zeros((N_STEPS + 1, NX))
U[0] = T.copy()

# Time stepping
for n in range(1, N_STEPS + 1):
    U[n] = ftcs(U[n-1], DT, DX, ALPHA)

In [ ]:
%matplotlib inline
# ----------------------------
# 3. 3D Surface Plot
# ----------------------------
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111, projection='3d')
X, T_grid = np.meshgrid(x, t)
surf = ax.plot_surface(X, T_grid, U, cmap='viridis', edgecolor='none', alpha=0.8)
ax.set_xlabel('Position x')
ax.set_ylabel('Time t')
ax.set_zlabel('Temperature T')
ax.set_title('1D Heat Equation (FTCS)')
fig.colorbar(surf, ax=ax, label='T')
plt.tight_layout()
plt.show()

In [ ]:
from IPython.display import HTML
# ----------------------------
# 4. Animation
# ----------------------------
fig_anim, ax_anim = plt.subplots(figsize=(8, 5))
line, = ax_anim.plot([], [], 'b-', lw=2.5)
ax_anim.set_xlim(0, L)
ax_anim.set_ylim(0, 120)
ax_anim.set_xlabel('Position x')
ax_anim.set_ylabel('Temperature T')
ax_anim.set_title('Heat Diffusion Animation')
time_text = ax_anim.text(0.02, 0.95, '', transform=ax_anim.transAxes, fontsize=12)

def init():
    line.set_data([], [])
    time_text.set_text('')
    return line, time_text

def animate(i):
    line.set_data(x, U[i])
    time_text.set_text(f't = {t[i]:.2f}')
    return line, time_text

anim = FuncAnimation(fig_anim, animate, frames=N_STEPS, init_func=init, interval=50, blit=True)
plt.show()

HTML(anim.to_jshtml())

In [ ]:
from IPython.display import HTML
HTML(anim.to_html5_video())


```{exercise} Explorations
- [ ] Play with $\lambda$. Is there really any instability?
- [x] Play with the initial conditions: is there any dependence?
- [x] Create an animation of the temperature time evolution.
```

### A note on derivative boundary conditions
In case you need to fix the derivative a the boundary, let's say an isolated wall on the left, where $\partial u / \partial x = c_1$, then you could introduce that as

\begin{equation}
u_0^{n+1} = u_0^n + \lambda(u_1^n - 2u_0^n + u_{-1}^n),
\end{equation}
where there is a new imaginary point at $i = -1$. But from the derivative condition you can get its value, since $(u_0 - u_{-1})/\Delta x \simeq \partial u/\partial x$.

## Implicit methods
Implicit methods improve the solution stability and, generally, result in in matrix systems that need to be solve. Examples can be found at https://github.com/numerical-mooc/numerical-mooc/blob/master/lessons/04_spreadout/04_02_Heat_Equation_1D_Implicit.ipynb . One of the most used implicit method is the [Crank-Nicholson method](https://en.wikipedia.org/wiki/Crank%E2%80%93Nicolson_method) is an implicit method which is commonly used given that it is unconditionally stable. It is second order in time, and its evolution is represented as

```{figure} https://upload.wikimedia.org/wikipedia/commons/thumb/1/1e/Crank-Nicolson-stencil.svg/250px-Crank-Nicolson-stencil.svg.png
:alt: 
:class: bg-primary
:width: 80%
:align: center
```
<!-- 

<center>
    <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/1/1e/Crank-Nicolson-stencil.svg/200px-Crank-Nicolson-stencil.svg.png" />
</center> -->

For more info check <https://github.com/numerical-mooc/numerical-mooc/tree/master/lessons/04_spreadout >

```{note} Limitations
They can be $O(N^3)$, or $O(N)$ with sparse solvers, but they are, usually, undconditionally stable. They are usually harder to implement, but the efforts pays. 
```


In [ ]:
# ==========================================
# IMPORTS & SETUP
# ==========================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve
from IPython.display import HTML
from matplotlib.animation import FuncAnimation

# Problem parameters (reuse your values)
L, NX, DX = 10.0, 51, 10.0/50
ALPHA = 0.6
N_STEPS = 400

# Grids
x = np.linspace(0, L, NX)
LAMBDA = 0.55
DT = LAMBDA * DX**2 / ALPHA  # Same DT as FTCS for fair comparison
t = np.arange(N_STEPS + 1) * DT

# Initial & Dirichlet BCs
T = np.zeros(NX)
T[0], T[-1] = 100.0, 50.0
T[1:-1] = 20.0

# Storage
U_cn = np.zeros((N_STEPS + 1, NX))
U_cn[0] = T.copy()

# ==========================================
# CRANK-NICOLSON SOLVER (Implicit)
# ==========================================
r = ALPHA * DT / DX**2

# Left-hand side: (I + r * Laplacian) → unconditionally stable
main_lhs = np.full(NX-2, 2 + 2*r)
off_lhs = np.full(NX-3, -r)
A = diags([off_lhs, main_lhs, off_lhs], [-1, 0, 1], shape=(NX-2, NX-2))

# Right-hand side: (I - r * Laplacian)
main_rhs = np.full(NX-2, 2 - 2*r)
off_rhs = np.full(NX-3, r)
B = diags([off_rhs, main_rhs, off_rhs], [-1, 0, 1], shape=(NX-2, NX-2))

for n in range(N_STEPS):
    rhs = B @ U_cn[n, 1:-1]
    # Dirichlet BCs contribute to RHS: r*(u^n + u^{n+1}) at boundaries
    rhs[0] += r * (U_cn[n, 0] + T[0])
    rhs[-1] += r * (U_cn[n, -1] + T[-1])
    
    U_cn[n+1, 1:-1] = spsolve(A, rhs)
    U_cn[n+1, 0] = T[0]
    U_cn[n+1, -1] = T[-1]

# ==========================================
# COMPARISON: FTCS (Explicit)
# ==========================================
U_ftcs = np.zeros_like(U_cn)
U_ftcs[0] = T.copy()
#LAMBDA = ALPHA * DT / DX**2

for n in range(N_STEPS):
    U_ftcs[n+1, 1:-1] = U_ftcs[n, 1:-1] + LAMBDA * (U_ftcs[n, 2:] - 2*U_ftcs[n, 1:-1] + U_ftcs[n, :-2])
    U_ftcs[n+1, 0] = T[0]
    U_ftcs[n+1, -1] = T[-1]

# ==========================================
# VISUALIZATION (Jupyter Book Safe)
# ==========================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
for ax, U, name in [(ax1, U_ftcs, "FTCS (Explicit)"), (ax2, U_cn, "Crank-Nicolson (Implicit)")]:
    ax.plot(x, U[0], 'k--', label='t=0', lw=2)
    ax.plot(x, U[-1], 'r-', label=f't={t[-1]:.2f}', lw=2)
    ax.set_title(f"{name} (LAMBDA={LAMBDA:.2f})")
    ax.set_xlabel('x'); ax.set_ylabel('T')
    ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()



## Parabolic equations in two dimensions
In this case you could be solving problems like

\begin{equation}
\frac{\partial T}{\partial t} = \alpha \left( \frac{\partial^2T}{\partial x^2} + \frac{\partial^2 T}{\partial y^2}\right).
\end{equation}

In this case, stability analysis implies that 

\begin{equation}
\Delta t \le \frac{1}{8} \frac{\Delta x^2 + \Delta y^2}{\alpha}.
\end{equation}

As with the previous 1D case, implicit schemes improve the stability , but the matrix systems are no longer tri-diagonal and therefore the matrix storage and computational times increase a lot.

There are alternative approaches , like the [ADI-Alternating direction implicit scheme](https://en.wikipedia.org/wiki/Alternating-direction_implicit_method) which can be use in very complex PDE, even non-linear.

In [ ]:
# ==========================================
# 2D HEAT EQUATION (FTCS) - PEDAGOGICAL EXAMPLE
# ==========================================
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# ----------------------------
# 1. Grid & Parameters
# ----------------------------
Lx, Ly = 1.0, 1.0
Nx, Ny = 60, 60
dx = Lx / (Nx - 1)
dy = Ly / (Ny - 1)
x = np.linspace(0, Lx, Nx)
y = np.linspace(0, Ly, Ny)

# 'ij' indexing ensures X[i,j] matches array layout u[i,j]
X, Y = np.meshgrid(x, y, indexing='ij')

alpha = 1.0
# 2D CFL condition: r_x + r_y <= 0.5
# For dx=dy => 2*r <= 0.5 => r <= 0.25
r = 0.2
dt = r * dx**2 / alpha
N_steps = 350
t = np.arange(N_steps + 1) * dt

# ----------------------------
# 2. IC & BCs
# ----------------------------
u = np.zeros((Nx, Ny))
x0, y0 = Lx/2, Ly/2
sigma = 0.15
# Gaussian pulse initial condition
u = np.exp(-((X - x0)**2 + (Y - y0)**2) / (2*sigma**2))

# Dirichlet BC: u=0 on all boundaries (already zero-initialized)
# Interior update naturally preserves BCs

# Storage for visualization
U = np.zeros((N_steps + 1, Nx, Ny))
U[0] = u.copy()

# ----------------------------
# 3. FTCS 2D Solver
# ----------------------------
for n in range(N_steps):
    # 5-point stencil: u_xx + u_yy
    u[1:-1, 1:-1] = u[1:-1, 1:-1] + r * (
        u[2:, 1:-1] - 2*u[1:-1, 1:-1] + u[:-2, 1:-1] +
        u[1:-1, 2:] - 2*u[1:-1, 1:-1] + u[1:-1, :-2]
    )
    U[n+1] = u.copy()

# ----------------------------
# 4. Static 3D Comparison (t=0 vs t=final)
# ----------------------------
fig_static = plt.figure(figsize=(10, 4.5))
ax1 = fig_static.add_subplot(121, projection='3d')
surf1 = ax1.plot_surface(X, Y, U[0], cmap='viridis', edgecolor='none', alpha=0.8)
ax1.set_title('t = 0.00')
ax1.set_xlabel('x'); ax1.set_ylabel('y'); ax1.set_zlabel('u')

ax2 = fig_static.add_subplot(122, projection='3d')
surf2 = ax2.plot_surface(X, Y, U[-1], cmap='viridis', edgecolor='none', alpha=0.8)
ax2.set_title(f't = {t[-1]:.3f}')
ax2.set_xlabel('x'); ax2.set_ylabel('y'); ax2.set_zlabel('u')
plt.tight_layout()
plt.show()

In [ ]:
# ----------------------------
# 5. Time Animation (2D Heatmap)
# ----------------------------
# pcolormesh expects X, Y to have shape (M+1, N+1) for data of shape (M, N)
x_vis = np.linspace(0, Lx, Nx + 1)
y_vis = np.linspace(0, Ly, Ny + 1)
X_vis, Y_vis = np.meshgrid(x_vis, y_vis, indexing='ij')

fig_anim, ax_anim = plt.subplots(figsize=(7, 5))
heatmap = ax_anim.pcolormesh(X_vis, Y_vis, U[0], cmap='viridis', shading='auto')
ax_anim.set_title('2D Heat Equation (FTCS)')
ax_anim.set_xlabel('x'); ax_anim.set_ylabel('y')
time_text = ax_anim.text(0.02, 0.95, '', transform=ax_anim.transAxes, fontsize=12, 
                         bbox=dict(facecolor='white', alpha=0.7))

def init():
    heatmap.set_array(U[0].ravel())
    time_text.set_text('')
    return heatmap, time_text

def animate(i):
    heatmap.set_array(U[i].ravel())
    # Use dynamic color scaling to prevent clipping if max temp changes
    vmax = max(np.max(U[0]), 1.0)
    heatmap.set_clim(vmin=0, vmax=vmax)
    time_text.set_text(f't = {t[i]:.3f}')
    return heatmap, time_text

anim = FuncAnimation(fig_anim, animate, frames=N_steps, init_func=init, 
                     interval=20, blit=False)
HTML(anim.to_html5_video())
#plt.close(fig_anim)


### Exercises
1. Solve the 1D heat equation with a Newmann condition at the left border , where the derivative is null.
2. Solve the heat equation in 2D using the Cranck-Nicholson method